# 04. Customer Retention & Cohort Analysis
This notebook calculates:
1. Customer Retention Rates
2. Repeat Purchase Distributions
3. Cohort Summaries based on the first purchase period

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

base_dir = pathlib.Path().resolve().parent
processed_dir = base_dir / 'data' / 'processed'
df_orders = pd.read_csv(processed_dir / 'processed_orders.csv')
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
print(df_orders.head())

## 1. Repeat Purchase Distribution

In [ ]:
orders_per_customer = df_orders.groupby('customer_unique_id')['order_id'].count().value_counts().sort_index()
print('Orders Per Customer Distribution:\n', orders_per_customer)

orders_per_customer.head(10).plot(kind='bar')
plt.xlabel('Number of Orders')
plt.ylabel('Number of Customers')
plt.title('Distribution of Orders per Customer')
plt.show()

## 2. Retention Rate Overview

In [ ]:
total_customers = df_orders['customer_unique_id'].nunique()
repeat_customers = df_orders[df_orders['repeat_customer_flag'] == 1]['customer_unique_id'].nunique()
overall_retention_rate = (repeat_customers / total_customers) * 100

print(f'Total Customers: {total_customers}')
print(f'Repeat Customers: {repeat_customers}')
print(f'Overall Retention/Repeat Purchase Rate: {overall_retention_rate:.2f}%')

## 3. Cohort Analysis

In [ ]:
# Determine first purchase month for each customer
df_orders['order_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')
df_orders['cohort_month'] = df_orders.groupby('customer_unique_id')['order_month'].transform('min')

# Calculate cohort sizes and purchase counts over time
cohort_data = df_orders.groupby(['cohort_month', 'order_month'])['customer_unique_id'].nunique().reset_index()

# Calculate the index (months since first purchase)
cohort_data['cohort_index'] = (cohort_data['order_month'] - cohort_data['cohort_month']).apply(lambda x: x.n)

cohort_pivot = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_unique_id')
cohort_sizes = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

print('Cohort Sizes:\n', cohort_sizes.head(10))
print('\nRetention Matrix Sample (Month 0-6 %):\n', retention_matrix.iloc[:10, :7])

plt.figure(figsize=(12, 8))
sns.heatmap(retention_matrix.iloc[:12, 1:12], annot=True, fmt='.1f', cmap='Blues')
plt.title('Cohort Customer Retention (%)')
plt.show()